# 01 – Data inspection and fixed patient split
Run top to bottom on a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).
This notebook loads the dataset, checks its format, recreates the seed-42 split and checks it against the committed `data/split_manifest.json`, and measures token lengths and truncation.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Look at the raw data

In [ ]:
from datasets import load_dataset
ds = load_dataset("ncbi/TrialGPT-Criterion-Annotations")
print(ds)
df = ds["train"].to_pandas()
print(df.columns.tolist())
print(df.expert_eligibility.value_counts(), "\n")
print(df.groupby(["criterion_type", "expert_eligibility"]).size(), "\n")
print("patients:", df.patient_id.nunique(), " trials:", df.trial_id.nunique())
print("records with empty expert evidence:", (df.expert_sentences == "[]").sum(), "/", len(df))
print("\nexample note:\n", df.note.iloc[0])
print("\nexample expert_sentences:", df.expert_sentences.iloc[:5].tolist())

## 2. Leakage control
These columns are **never** used as input or target:
- `gpt4_explanation`, `gpt4_sentences`, `gpt4_eligibility`: GPT-4 outputs
- `explanation_correctness`: an expert grade of GPT-4's explanation, which indirectly leaks the GPT-4 output
- `training`: TrialGPT's own split flag. We use the prescribed seed-42 split instead.

`src/data.py` keeps only `KEEP_COLUMNS`.

In [ ]:
import data, json
print("dropped:", [c for c in df.columns if c not in data.KEEP_COLUMNS])
rows = data.load_rows()
print(len(rows), "rows; keys:", list(rows[0].keys()))
# every note is numbered 0..n-1 and every evidence ID exists in its note
for r in rows:
    ids = [i for i, _ in r["sentences"]]
    assert ids == list(range(len(ids))), r["annotation_id"]
    assert set(r["expert_sentences"]) <= set(ids), r["annotation_id"]
print("sentence-ID checks passed")

## 3. Fixed split (37 / 8 / 8 patients, `default_rng(42)`)

In [ ]:
manifest = data.build_manifest(rows)            # sorts IDs, shuffles with default_rng(42), asserts no leakage
committed = json.load(open("data/split_manifest.json"))
for k in ["train", "val", "test"]:
    assert manifest[k] == committed[k], f"{k} differs from committed manifest!"
print("Recomputed split == committed manifest  ✔")
for k in ["train", "val", "test"]:
    c = manifest["counts"][k]
    print(f"{k:5s} patients={c['patients']:2d} annotations={c['annotations']:3d} "
          f"incl={c['inclusion']:3d} excl={c['exclusion']:3d} empty_evidence={c['empty_evidence']:3d}")
    print("      ", c["labels"])
print("\nval patients :", manifest["val"])
print("test patients:", manifest["test"])

## 4. Token lengths and truncation (Qwen2.5 tokenizer)

In [ ]:
from transformers import AutoTokenizer
import numpy as np
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
def n_tokens(msgs, gen_prompt):
    return len(tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen_prompt),
                   add_special_tokens=False)["input_ids"])
prompt_len = np.array([n_tokens(data.row_messages(r), True) for r in rows])
full_len   = np.array([n_tokens(data.row_messages(r, with_target=True), False) for r in rows])
target_len = full_len - prompt_len
MAX_LEN = 1536
stats = {
  "prompt_tokens": {"min": int(prompt_len.min()), "mean": float(prompt_len.mean()), "p95": float(np.percentile(prompt_len, 95)), "max": int(prompt_len.max())},
  "target_tokens": {"min": int(target_len.min()), "mean": float(target_len.mean()), "max": int(target_len.max())},
  "max_len_used": MAX_LEN,
  "examples_over_max_len": int((full_len > MAX_LEN).sum()),
  "note_sentences_max": max(len(r["sentences"]) for r in rows),
  "qwen_context_window": 32768,
}
print(json.dumps(stats, indent=2))
assert stats["examples_over_max_len"] == 0, "some examples would be truncated, so evidence could be lost"
json.dump(stats, open("results/token_lengths.json", "w"), indent=2)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,3.5)); plt.hist(prompt_len, bins=40)
plt.axvline(MAX_LEN, color="r", ls="--", label=f"max_len={MAX_LEN}")
plt.xlabel("prompt tokens (chat template)"); plt.ylabel("annotations"); plt.legend(); plt.tight_layout()
plt.savefig("results/token_length_hist.png", dpi=150); plt.show()

## 5. Save environment and push

In [ ]:
!pip freeze > logs/colab_env_01.txt
!git add data/split_manifest.json results/token_lengths.json results/token_length_hist.png logs/colab_env_01.txt
!git commit -m "Verify seed-42 split, token-length/truncation analysis" && git push